# NFL Upset Prediction: XGBoost Model

This notebook trains and evaluates the XGBoost model:
1. Load processed features
2. Time-series cross-validation
3. Hyperparameter tuning with Optuna
4. SHAP feature importance analysis
5. Save trained model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import optuna
import joblib
from sklearn.metrics import roc_auc_score, brier_score_loss, classification_report

# Project imports
import sys
sys.path.insert(0, '..')

from src.models.xgboost_model import UpsetXGBoost
from src.models.cv_splitter import TimeSeriesCVSplitter
from src.models.trainer import ModelTrainer
from src.evaluation.metrics import EvaluationMetrics
from src.evaluation.shap_analysis import compute_shap_values, get_shap_feature_importance

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

# Reproducibility
np.random.seed(42)

## 1. Load Processed Data

In [ ]:
# Load engineered features
df = pd.read_parquet('../data/processed/features_engineered.parquet')
print(f"Loaded {len(df):,} records")

# Load feature columns
with open('../data/processed/feature_columns.txt', 'r') as f:
    feature_columns = [line.strip() for line in f.readlines()]
print(f"Feature columns: {len(feature_columns)}")

In [ ]:
# Prepare X and y
# Drop rows with missing features
df_clean = df.dropna(subset=feature_columns + ['is_upset'])
print(f"Records after dropping missing: {len(df_clean):,}")

X = df_clean[feature_columns].copy()
y = df_clean['is_upset'].astype(int)

print(f"\nFeatures shape: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"Upset rate: {y.mean():.1%}")

## 2. Time-Series Cross-Validation

In [ ]:
# Setup time-series CV
cv_splitter = TimeSeriesCVSplitter(n_folds=3)

# Train with default parameters first
model = UpsetXGBoost()

# Manual CV loop to track per-fold metrics
fold_metrics = []
fold_predictions = []

for fold_idx, (train_idx, val_idx) in enumerate(cv_splitter.split(df_clean)):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]
    
    print(f"\nFold {fold_idx + 1}:")
    print(f"  Train: {len(X_train):,} games, Val: {len(X_val):,} games")
    print(f"  Train seasons: {df_clean.iloc[train_idx]['season'].min()}-{df_clean.iloc[train_idx]['season'].max()}")
    print(f"  Val season: {df_clean.iloc[val_idx]['season'].unique()[0]}")
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict_proba(X_val)
    
    # Evaluate
    metrics = EvaluationMetrics(y_val, y_pred)
    fold_result = {
        'fold': fold_idx + 1,
        'auc_roc': metrics.auc_roc(),
        'brier_score': metrics.brier_score(),
        'accuracy': metrics.accuracy(),
    }
    fold_metrics.append(fold_result)
    fold_predictions.append((val_idx, y_val.values, y_pred))
    
    print(f"  AUC-ROC: {fold_result['auc_roc']:.4f}")
    print(f"  Brier Score: {fold_result['brier_score']:.4f}")

In [ ]:
# Summary of CV results
cv_results = pd.DataFrame(fold_metrics)
print("\nCross-Validation Results:")
print(cv_results.to_string(index=False))
print(f"\nMean AUC-ROC: {cv_results['auc_roc'].mean():.4f} (+/- {cv_results['auc_roc'].std():.4f})")
print(f"Mean Brier Score: {cv_results['brier_score'].mean():.4f} (+/- {cv_results['brier_score'].std():.4f})")

## 3. Hyperparameter Tuning with Optuna

In [ ]:
def objective(trial):
    """Optuna objective for XGBoost hyperparameter tuning."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
    }
    
    model = UpsetXGBoost(**params)
    
    # Cross-validation
    cv_scores = []
    for train_idx, val_idx in cv_splitter.split(df_clean):
        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)
        cv_scores.append(roc_auc_score(y_val, y_pred))
    
    return np.mean(cv_scores)

# Run optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest AUC-ROC: {study.best_value:.4f}")
print(f"Best parameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [ ]:
# Plot optimization history
fig, ax = plt.subplots(figsize=(10, 4))
trials_df = study.trials_dataframe()
ax.plot(trials_df['number'], trials_df['value'], 'o-', alpha=0.5)
ax.axhline(y=study.best_value, color='red', linestyle='--', label=f'Best: {study.best_value:.4f}')
ax.set_xlabel('Trial')
ax.set_ylabel('AUC-ROC')
ax.set_title('Optuna Optimization History')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/xgboost_optuna_history.png', dpi=150)
plt.show()

## 4. Train Final Model

In [ ]:
# Train final model with best parameters on all data
best_model = UpsetXGBoost(**study.best_params)
best_model.fit(X, y)

# Get XGBoost feature importance
xgb_importance = best_model.feature_importance()
importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance': [xgb_importance.get(f, 0) for f in feature_columns]
}).sort_values('importance', ascending=False)

print("Top 15 XGBoost Feature Importances:")
print(importance_df.head(15).to_string(index=False))

## 5. SHAP Analysis

In [ ]:
# Compute SHAP values
shap_values = compute_shap_values(best_model, X)

# SHAP feature importance
shap_importance = get_shap_feature_importance(best_model, X)
shap_df = pd.DataFrame({
    'feature': list(shap_importance.keys()),
    'shap_importance': list(shap_importance.values())
}).sort_values('shap_importance', ascending=False)

print("Top 15 SHAP Feature Importances:")
print(shap_df.head(15).to_string(index=False))

In [ ]:
# SHAP summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X, show=False, max_display=20)
plt.tight_layout()
plt.savefig('../results/figures/xgboost_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP bar plot
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X, plot_type='bar', show=False, max_display=15)
plt.tight_layout()
plt.savefig('../results/figures/xgboost_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare XGBoost vs SHAP importance
comparison = importance_df.merge(shap_df, on='feature')
comparison['xgb_rank'] = comparison['importance'].rank(ascending=False)
comparison['shap_rank'] = comparison['shap_importance'].rank(ascending=False)

print("Feature Importance Comparison (XGBoost vs SHAP):")
print(comparison.sort_values('shap_rank').head(15).to_string(index=False))

## 6. Save Model

In [ ]:
# Save model and metadata
model_path = '../results/models/xgboost_upset_model.joblib'
import os
os.makedirs('../results/models', exist_ok=True)

model_data = {
    'model': best_model,
    'params': study.best_params,
    'feature_columns': feature_columns,
    'cv_auc': study.best_value,
    'shap_importance': shap_importance,
    'xgb_importance': xgb_importance,
}
joblib.dump(model_data, model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Final summary
print("=" * 50)
print("XGBOOST MODEL SUMMARY")
print("=" * 50)
print(f"\nCross-validation AUC-ROC: {study.best_value:.4f}")
print(f"\nTop 5 features (SHAP):")
for i, row in shap_df.head(5).iterrows():
    print(f"  {row['feature']}: {row['shap_importance']:.4f}")
print(f"\nModel saved to: results/models/xgboost_upset_model.joblib")